# Notebook 2: Evaluation on a New Dataset

This notebook:
- Loads the fine-tuned Whisper model saved in Notebook 1
- Runs inference on `test-clean` (or any custom folder you specify)
- Computes **Word Error Rate (WER)** and **Character Error Rate (CER)**
- Displays a comparison table of predicted vs. ground-truth transcripts

## Step 1: Install Dependencies

In [ ]:
!pip install transformers torch torchaudio evaluate jiwer soundfile librosa pandas

## Step 2: Imports and Paths

In [ ]:
import torch
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import Dataset, Audio
import evaluate

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path("../../../")                                       # NLP_project root
WEIGHTS_DIR = Path("../weights/whisper-base-librispeech")             # saved model
EXTRACT_DIR = BASE_DIR / "librispeech"                                # extracted audio

# Change this to any folder that contains .flac/.wav files + *.trans.txt
EVAL_SPLIT  = "test-clean"

SAMPLE_RATE = 16_000
MAX_SAMPLES = 200   # set None to evaluate all

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 3: Load Fine-Tuned Model

In [ ]:
print(f"Loading model from: {WEIGHTS_DIR.resolve()}")
processor = WhisperProcessor.from_pretrained(str(WEIGHTS_DIR))
model     = WhisperForConditionalGeneration.from_pretrained(str(WEIGHTS_DIR)).to(device)
model.eval()
print("Model loaded successfully.")

## Step 4: Load Evaluation Dataset

In [ ]:
def parse_librispeech_split(root: Path, split_name: str) -> Dataset:
    """Collect (audio_path, transcript) pairs from a LibriSpeech split."""
    audio_paths, transcripts = [], []
    split_dir = root / "LibriSpeech" / split_name
    if not split_dir.exists():
        raise FileNotFoundError(f"Split not found: {split_dir}")
    for trans_file in sorted(split_dir.rglob("*.trans.txt")):
        chapter_dir = trans_file.parent
        with open(trans_file) as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                if len(parts) != 2:
                    continue
                utt_id, text = parts
                flac_path = chapter_dir / f"{utt_id}.flac"
                if flac_path.exists():
                    audio_paths.append(str(flac_path))
                    transcripts.append(text.lower())
    ds = Dataset.from_dict({"audio": audio_paths, "sentence": transcripts})
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return ds

eval_dataset = parse_librispeech_split(EXTRACT_DIR, EVAL_SPLIT)
if MAX_SAMPLES:
    eval_dataset = eval_dataset.select(range(min(MAX_SAMPLES, len(eval_dataset))))

print(f"Evaluation samples: {len(eval_dataset)}")

## Step 5: Run Inference

In [ ]:
def transcribe_batch(audio_arrays: list) -> list:
    """Run Whisper inference on a list of 16 kHz audio arrays."""
    inputs = processor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(inputs)

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)


predictions, references = [], []
BATCH_SIZE = 8

for i in range(0, len(eval_dataset), BATCH_SIZE):
    batch = eval_dataset[i : i + BATCH_SIZE]
    audio_arrays = [a["array"] for a in batch["audio"]]
    preds = transcribe_batch(audio_arrays)
    predictions.extend(preds)
    references.extend(batch["sentence"])
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Processed {min(i + BATCH_SIZE, len(eval_dataset))} / {len(eval_dataset)}")

print("Inference complete.")

## Step 6: Compute WER and CER

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

wer = wer_metric.compute(predictions=predictions, references=references)
cer = cer_metric.compute(predictions=predictions, references=references)

print(f"\n{'='*40}")
print(f"  Word Error Rate  (WER) : {wer:.4f}  ({wer*100:.2f}%)")
print(f"  Char Error Rate  (CER) : {cer:.4f}  ({cer*100:.2f}%)")
print(f"{'='*40}")

## Step 7: Show Example Predictions vs. Ground Truth

In [ ]:
N_EXAMPLES = 10

results_df = pd.DataFrame({
    "Ground Truth" : references[:N_EXAMPLES],
    "Prediction"   : predictions[:N_EXAMPLES],
})

# Per-sample WER for context
results_df["Sample WER"] = results_df.apply(
    lambda row: round(
        wer_metric.compute(predictions=[row["Prediction"]], references=[row["Ground Truth"]]), 4
    ),
    axis=1,
)

pd.set_option("display.max_colwidth", 80)
display(results_df)

## Step 8 (Optional): Evaluate on a Custom Folder

Set `CUSTOM_AUDIO_DIR` to any folder containing `.flac` or `.wav` files. Transcripts must be in `*.trans.txt` files in the same LibriSpeech format.

In [ ]:
# ── Uncomment and edit this cell to run on a custom folder ─────────────────
# CUSTOM_AUDIO_DIR = Path("/path/to/your/custom/audio")
# custom_ds = parse_librispeech_split(CUSTOM_AUDIO_DIR.parent, CUSTOM_AUDIO_DIR.name)
# custom_preds, custom_refs = [], []
# for i in range(0, len(custom_ds), BATCH_SIZE):
#     batch = custom_ds[i : i + BATCH_SIZE]
#     custom_preds.extend(transcribe_batch([a["array"] for a in batch["audio"]]))
#     custom_refs.extend(batch["sentence"])
# custom_wer = wer_metric.compute(predictions=custom_preds, references=custom_refs)
# custom_cer = cer_metric.compute(predictions=custom_preds, references=custom_refs)
# print(f"Custom WER: {custom_wer:.4f} | Custom CER: {custom_cer:.4f}")